# 11 — Enterprise Performance Optimization & Maintenance

| Section | Topic |
|---------|-------|
| 1 | Setup & Configuration |
| 2 | Build Optimized Tables — Liquid Clustering vs Partitioned + Z-Order |
| 3 | Benchmarking — 5 query patterns, 3 runs each |
| 4 | MERGE INTO — Late-Arriving Data Upserts |
| 5 | Churn Metrics — Stale Inventory & Brand Activity |
| 6 | Delta Maintenance — VACUUM + OPTIMIZE |
| 7 | Final Summary & Strategy Comparison |

## Section 1 — Setup & Configuration

In [0]:
import time
from pyspark.sql import functions as F

dbutils.widgets.text("project_catalog", "vstone_catalog")
dbutils.widgets.text("gold_schema",     "gold")
dbutils.widgets.text("silver_schema",   "silver")

CATALOG = dbutils.widgets.get("project_catalog")
GOLD    = dbutils.widgets.get("gold_schema")
SILVER  = dbutils.widgets.get("silver_schema")

FACT        = f"{CATALOG}.{GOLD}.fact_listings"
FACT_LIQUID = f"{CATALOG}.{GOLD}.fact_listings_liquid"
FACT_PART   = f"{CATALOG}.{GOLD}.fact_listings_partitioned"
BENCH_TABLE = f"{CATALOG}.{GOLD}.benchmark_results"
CHURN_TABLE = f"{CATALOG}.{GOLD}.agg_stale_inventory"
BRAND_CHURN = f"{CATALOG}.{GOLD}.agg_brand_churn_metrics"

# Dim tables for label resolution
DIM_CAR      = f"{CATALOG}.{GOLD}.dim_car"
DIM_LOCATION = f"{CATALOG}.{GOLD}.dim_location"
DIM_PRICE    = f"{CATALOG}.{GOLD}.dim_price_category"
DIM_STEERING = f"{CATALOG}.{GOLD}.dim_steering"

print("=" * 70)
print("  PERFORMANCE OPTIMIZATION SUITE — VStone Gold Layer")
print("=" * 70)
print(f"  Source fact   : {FACT}")
print(f"  Liquid table  : {FACT_LIQUID}")
print(f"  Partitioned   : {FACT_PART}")
print(f"  Benchmark     : {BENCH_TABLE}")
print(f"  Churn metric  : {CHURN_TABLE}")
print(f"  Brand churn   : {BRAND_CHURN}")
print("=" * 70)

fact_rows = spark.table(FACT).count()
print(f"\n  fact_listings rows : {fact_rows:,}")
print("  Ready.")

## Section 2 — Build Optimized Tables

### Strategy A: Liquid Clustering
Cluster keys use **integer surrogate keys** (`car_sk`, `location_sk`) — not strings.
Integer clustering is more efficient: smaller sort keys, faster file pruning.

### Strategy B: Traditional Partitioning + Z-Order
Partition by `listing_year` (always present, low cardinality, even distribution).
Z-Order by `car_sk` and `location_sk` for multi-dimensional file pruning.

In [0]:
# ── APPROACH A: LIQUID CLUSTERING ───────────────────────────────────────────
print("━" * 70)
print("  APPROACH A: Liquid Clustering")
print("━" * 70)

spark.sql(f"""
CREATE OR REPLACE TABLE {FACT_LIQUID}
USING DELTA
CLUSTER BY (car_sk, listing_year, location_sk, price_category_key)
TBLPROPERTIES (
  'quality'                    = 'gold',
  'optimization'               = 'liquid_clustering',
  'cluster_keys'               = 'car_sk_listing_year_location_sk_price_category_key',
  'delta.enableChangeDataFeed' = 'true'
)
AS SELECT * FROM {FACT}
""")

spark.sql(f"OPTIMIZE {FACT_LIQUID}")
liq_rows = spark.table(FACT_LIQUID).count()
print(f"  Liquid table ready — {liq_rows:,} rows")
print("  Cluster keys: car_sk | listing_year | location_sk | price_category_key")


# ── APPROACH B: PARTITIONING + Z-ORDER ──────────────────────────────────────
print("━" * 70)
print("  APPROACH B: Partitioning + Z-Order")
print("━" * 70)

spark.sql(f"""
CREATE OR REPLACE TABLE {FACT_PART}
USING DELTA
PARTITIONED BY (listing_year)
TBLPROPERTIES (
  'quality'                    = 'gold',
  'optimization'               = 'partition_zorder',
  'partition_key'              = 'listing_year',
  'zorder_keys'                = 'car_sk_location_sk',
  'delta.enableChangeDataFeed' = 'true'
)
AS SELECT * FROM {FACT}
""")

spark.sql(f"OPTIMIZE {FACT_PART} ZORDER BY (car_sk, location_sk)")

part_rows = spark.table(FACT_PART).count()
print(f"  Partitioned table ready — {part_rows:,} rows")
print("  Partition: listing_year  |  Z-Order: car_sk, location_sk")

print("\n  Partition distribution (listing_year):")
spark.table(FACT_PART).groupBy("listing_year").count().orderBy("listing_year").show(25, truncate=False)

## Section 3 — Benchmarking Framework

**Method:** Each query runs 3 times. Average execution time is recorded.

| Query | Pattern | Column used |
|-------|---------|-------------|
| Q1 | Single car_sk filter | `car_sk` (INT surrogate key) |
| Q2 | Multi-car_sk + mileage range | `car_sk IN (...)` + range scan |
| Q3 | location_sk + price_category_key | Two INT FK filters |
| Q4 | Full aggregation by car_sk | Scan-heavy analytics |
| Q5 | listing_year range | Partition key home turf |


In [0]:
def benchmark(query, label, runs=3):
    times = []
    for i in range(runs):
        start   = time.perf_counter()
        spark.sql(query).collect()
        elapsed = round(time.perf_counter() - start, 3)
        times.append(elapsed)
        print(f"      Run {i+1}: {elapsed}s")
    avg = round(sum(times) / len(times), 3)
    print(f"    [{label}] Average: {avg}s")
    return avg


QUERIES = [
    (
        "Q1: Single brand filter (via dim_car join)",
        f"""SELECT f.listing_id, d.brand, d.model, f.price_rub
           FROM {FACT_LIQUID} f
           JOIN {DIM_CAR} d ON f.car_sk = d.car_sk AND d.__END_AT IS NULL
           WHERE d.brand = 'toyota'""",
        f"""SELECT f.listing_id, d.brand, d.model, f.price_rub
           FROM {FACT_PART} f
           JOIN {DIM_CAR} d ON f.car_sk = d.car_sk AND d.__END_AT IS NULL
           WHERE d.brand = 'toyota'""",
    ),
    (
        "Q2: Multi-brand + mileage range (car_sk filter)",
        f"""SELECT f.car_sk, COUNT(*), AVG(f.price_rub)
           FROM {FACT_LIQUID} f
           JOIN {DIM_CAR} d ON f.car_sk = d.car_sk AND d.__END_AT IS NULL
           WHERE d.brand IN ('toyota','honda','kia') AND f.mileage_km < 50000
           GROUP BY f.car_sk""",
        f"""SELECT f.car_sk, COUNT(*), AVG(f.price_rub)
           FROM {FACT_PART} f
           JOIN {DIM_CAR} d ON f.car_sk = d.car_sk AND d.__END_AT IS NULL
           WHERE d.brand IN ('toyota','honda','kia') AND f.mileage_km < 50000
           GROUP BY f.car_sk""",
    ),
    (
        "Q3: location_sk + price_category_key (INT FK filters)",
        f"""SELECT f.location_sk, f.price_category_key, COUNT(*), AVG(f.price_usd)
           FROM {FACT_LIQUID} f
           WHERE f.price_category_key = 4
           GROUP BY f.location_sk, f.price_category_key""",
        f"""SELECT f.location_sk, f.price_category_key, COUNT(*), AVG(f.price_usd)
           FROM {FACT_PART} f
           WHERE f.price_category_key = 4
           GROUP BY f.location_sk, f.price_category_key""",
    ),
    (
        "Q4: Full aggregation by brand (scan-heavy, dim_car join)",
        f"""SELECT d.brand, COUNT(*) AS listings, AVG(f.price_rub) AS avg_price
           FROM {FACT_LIQUID} f
           JOIN {DIM_CAR} d ON f.car_sk = d.car_sk AND d.__END_AT IS NULL
           GROUP BY d.brand ORDER BY listings DESC""",
        f"""SELECT d.brand, COUNT(*) AS listings, AVG(f.price_rub) AS avg_price
           FROM {FACT_PART} f
           JOIN {DIM_CAR} d ON f.car_sk = d.car_sk AND d.__END_AT IS NULL
           GROUP BY d.brand ORDER BY listings DESC""",
    ),
    (
        "Q5: listing_year range (partition key home turf)",
        f"""SELECT f.listing_year, COUNT(*), SUM(f.price_rub)
           FROM {FACT_LIQUID} f
           WHERE f.listing_year BETWEEN 2020 AND 2023
           GROUP BY f.listing_year""",
        f"""SELECT f.listing_year, COUNT(*), SUM(f.price_rub)
           FROM {FACT_PART} f
           WHERE f.listing_year BETWEEN 2020 AND 2023
           GROUP BY f.listing_year""",
    ),
]

benchmark_results = []
print(f"  BENCHMARKING {len(QUERIES)} QUERY PATTERNS (3 runs each)")
print("=" * 70)

for label, q_liq, q_part in QUERIES:
    print(f"\n  {label}")
    print("  Liquid Clustering:")
    t_liq  = benchmark(q_liq,  "Liquid",  runs=3)
    print("  Partitioned + Z-Order:")
    t_part = benchmark(q_part, "Z-Order", runs=3)
    winner  = "Liquid" if t_liq <= t_part else "Z-Order"
    speedup = round(max(t_liq, t_part) / min(t_liq, t_part), 2) if min(t_liq, t_part) > 0 else 1.0
    print(f"  Winner: {winner}  ({speedup}x faster)")
    benchmark_results.append((label, float(t_liq), float(t_part), winner, speedup))

In [0]:
schema = "query STRING, liquid_avg_secs DOUBLE, zorder_avg_secs DOUBLE, winner STRING, speedup DOUBLE"
spark.createDataFrame(benchmark_results, schema) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(BENCH_TABLE)

spark.sql(f'COMMENT ON TABLE {BENCH_TABLE} IS "Benchmark: Liquid vs Partitioned+Z-Order, 5 queries x 3 runs"')
print("  Benchmark results saved.")
display(spark.table(BENCH_TABLE).orderBy("query"))

liquid_wins = sum(1 for r in benchmark_results if r[3] == "Liquid")
zorder_wins = len(benchmark_results) - liquid_wins
print(f"\n  Liquid wins : {liquid_wins}/{len(benchmark_results)}")
print(f"  Z-Order wins: {zorder_wins}/{len(benchmark_results)}")

## Section 4 — MERGE INTO: Late-Arriving Data Upserts

In [0]:
# ── CLEANUP PREVIOUS DEMO DATA ──────────────────────────────────────────────
print(f"Cleaning up previous demo records from {FACT_LIQUID}...")
spark.sql(f"DELETE FROM {FACT_LIQUID} WHERE listing_id LIKE 'NEW_%'")
before_count = spark.table(FACT_LIQUID).count()
print(f"  Rows BEFORE merge: {before_count:,}")


# ── BUILD LATE-ARRIVING BATCH ────────────────────────────────────────────────
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW late_arriving_batch AS
SELECT * FROM (
    SELECT
        listing_id,
        listing_date,
        car_sk,
        location_sk,
        CASE
            WHEN ROUND(price_rub * 0.95, 2) < 300000                           THEN 1
            WHEN ROUND(price_rub * 0.95, 2) BETWEEN 300000  AND 700000          THEN 2
            WHEN ROUND(price_rub * 0.95, 2) BETWEEN 700001  AND 1500000         THEN 3
            WHEN ROUND(price_rub * 0.95, 2) > 1500000                          THEN 4
            ELSE 5
        END                                        AS price_category_key,
        steering_key,
        manufacture_year,
        engine_power,
        mileage_km,
        has_license,
        listing_year,
        listing_month,
        car_age_years,
        ROUND(price_rub * 0.95, 2)                 AS price_rub,
        ROUND(price_rub * 0.95 / 82.5, 2)          AS price_usd,
        car_age_at_listing,
        is_high_mileage,
        ROUND(price_rub * 0.95 / 82.5 / NULLIF(engine_power, 0), 2) AS price_per_hp_usd,
        photo_count,
        word_count,
        color_r, color_g, color_b,
        bronze_load_dt, bronze_source_file, silver_load_dt,
        current_timestamp()                        AS gold_load_dt
    FROM {FACT}
    LIMIT 500
)
UNION ALL
SELECT * FROM (
    SELECT
        CONCAT('NEW_', CAST(rn AS STRING))         AS listing_id,
        CAST(current_timestamp() AS DATE)           AS listing_date,
        car_sk,
        location_sk,
        price_category_key,
        steering_key,
        manufacture_year,
        engine_power,
        mileage_km,
        has_license,
        listing_year,
        listing_month,
        car_age_years,
        price_rub,
        price_usd,
        car_age_at_listing,
        is_high_mileage,
        price_per_hp_usd,
        photo_count,
        word_count,
        color_r, color_g, color_b,
        bronze_load_dt, bronze_source_file, silver_load_dt,
        current_timestamp() AS gold_load_dt
    FROM (
        SELECT *, ROW_NUMBER() OVER (ORDER BY listing_id) AS rn
        FROM {FACT}
        LIMIT 10
    )
)
""")
print("  Late-arriving batch view created (500 updates + 10 new inserts).")

In [0]:
# ── EXECUTE MERGE & VERIFY ────────────────────────────────────────────────────
print(f"Executing MERGE INTO {FACT_LIQUID}...")

spark.sql(f"""
MERGE INTO {FACT_LIQUID} AS target
USING late_arriving_batch AS source
ON target.listing_id = source.listing_id

WHEN MATCHED AND target.price_rub != source.price_rub THEN
  UPDATE SET
    target.price_rub          = source.price_rub,
    target.price_usd          = source.price_usd,
    target.price_category_key = source.price_category_key,
    target.price_per_hp_usd   = source.price_per_hp_usd,
    target.is_high_mileage    = source.is_high_mileage,
    target.gold_load_dt       = source.gold_load_dt

WHEN NOT MATCHED THEN INSERT (
    listing_id, listing_date, car_sk, location_sk,
    price_category_key, steering_key,
    manufacture_year, engine_power, mileage_km, has_license,
    listing_year, listing_month, car_age_years,
    price_rub, price_usd,
    car_age_at_listing, is_high_mileage, price_per_hp_usd,
    photo_count, word_count,
    color_r, color_g, color_b,
    bronze_load_dt, bronze_source_file, silver_load_dt, gold_load_dt
) VALUES (
    source.listing_id, source.listing_date, source.car_sk, source.location_sk,
    source.price_category_key, source.steering_key,
    source.manufacture_year, source.engine_power, source.mileage_km, source.has_license,
    source.listing_year, source.listing_month, source.car_age_years,
    source.price_rub, source.price_usd,
    source.car_age_at_listing, source.is_high_mileage, source.price_per_hp_usd,
    source.photo_count, source.word_count,
    source.color_r, source.color_g, source.color_b,
    source.bronze_load_dt, source.bronze_source_file, source.silver_load_dt, source.gold_load_dt
)
""")

after_count = spark.table(FACT_LIQUID).count()
print(f"\n  Rows BEFORE : {before_count:,}")
print(f"  Rows AFTER  : {after_count:,}")
print(f"  Net New     : {after_count - before_count:,}  (Expected: 10)")

display(spark.sql(f'SELECT listing_id, car_sk, price_rub, gold_load_dt FROM {FACT_LIQUID} WHERE listing_id LIKE "NEW_%"'))

## Section 5 — Churn Metrics

**Two churn tables:**
1. `agg_stale_inventory` — grain: `car_sk + location_sk`, with `churn_status` label
2. `agg_brand_churn_metrics` — grain: `brand` (resolved via `dim_car`), with `churn_score_pct`

In [0]:
# ── CHURN METRIC 1: STALE INVENTORY ────────────────────────────────────────
print("  Building Stale Inventory churn metric (>180 days inactive)...")

# Grain: car_sk + location_sk (integer keys from fact_listings)
# brand/model/fuel_type resolved via dim_car join at the end
spark.sql(f"""
CREATE OR REPLACE TABLE {CHURN_TABLE}
USING DELTA
TBLPROPERTIES (
  'quality'           = 'gold',
  'metric_type'       = 'churn',
  'churn_threshold'   = '180_days',
  'churn_grain'       = 'car_sk_location_sk',
  'refresh_frequency' = 'daily'
)
AS
WITH daily_activity AS (
    SELECT
        car_sk,
        location_sk,
        CAST(listing_date AS DATE)  AS listing_day,
        COUNT(*)                    AS daily_listings,
        AVG(price_rub)              AS avg_price_rub,
        AVG(price_usd)              AS avg_price_usd
    FROM {FACT_LIQUID}
    WHERE listing_date IS NOT NULL
    GROUP BY car_sk, location_sk, CAST(listing_date AS DATE)
),
last_seen AS (
    SELECT
        car_sk,
        location_sk,
        MAX(listing_day)            AS last_listing_date,
        MIN(listing_day)            AS first_listing_date,
        COUNT(DISTINCT listing_day) AS active_days,
        SUM(daily_listings)         AS total_listings,
        ROUND(AVG(avg_price_rub),0) AS avg_price_rub,
        ROUND(AVG(avg_price_usd),2) AS avg_price_usd
    FROM daily_activity
    GROUP BY car_sk, location_sk
)
SELECT
    ls.car_sk,
    ls.location_sk,
    dc.brand,
    dc.model,
    dl.city_name,
    ls.last_listing_date,
    ls.first_listing_date,
    ls.active_days,
    ls.total_listings,
    ls.avg_price_rub,
    ls.avg_price_usd,
    DATEDIFF(current_date(), ls.last_listing_date) AS days_inactive,
    CASE
        WHEN DATEDIFF(current_date(), ls.last_listing_date) > 365 THEN 'CRITICAL'
        WHEN DATEDIFF(current_date(), ls.last_listing_date) > 180 THEN 'STALE'
        WHEN DATEDIFF(current_date(), ls.last_listing_date) > 90  THEN 'AT_RISK'
        ELSE 'ACTIVE'
    END AS churn_status,
    current_timestamp() AS metric_computed_at
FROM last_seen ls
LEFT JOIN {DIM_CAR}      dc ON ls.car_sk      = dc.car_sk      AND dc.__END_AT IS NULL
LEFT JOIN {DIM_LOCATION} dl ON ls.location_sk  = dl.location_sk AND dl.__END_AT IS NULL
WHERE DATEDIFF(current_date(), ls.last_listing_date) > 180
""")

spark.sql(f'COMMENT ON TABLE {CHURN_TABLE} IS "Churn: car_sk+location_sk combos inactive >180 days"')
stale_cnt = spark.table(CHURN_TABLE).count()
print(f"  Stale records: {stale_cnt:,} car+location combos")
print("\n  Churn status breakdown:")
spark.table(CHURN_TABLE).groupBy("churn_status").count().orderBy(F.desc("count")).show(truncate=False)

In [0]:
# ── CHURN METRIC 2: BRAND CHURN SCORE ───────────────────────────────────────
print("  Building Brand Churn Score metric...")

# Brand label resolved from dim_car -- not from fact_listings (no brand column there)
spark.sql(f"""
CREATE OR REPLACE TABLE {BRAND_CHURN}
USING DELTA
TBLPROPERTIES (
  'quality'           = 'gold',
  'metric_type'       = 'churn',
  'churn_grain'       = 'brand',
  'refresh_frequency' = 'daily'
)
AS
WITH brand_totals AS (
    SELECT
        dc.brand,
        COUNT(DISTINCT CONCAT(CAST(f.car_sk AS STRING), '_', CAST(f.location_sk AS STRING)))
                                                          AS total_combos,
        COUNT(DISTINCT f.car_sk)                          AS distinct_car_sks,
        COUNT(DISTINCT f.location_sk)                     AS distinct_locations,
        COUNT(*)                                          AS total_listings,
        ROUND(AVG(f.price_rub), 0)                        AS avg_price_rub,
        ROUND(AVG(f.mileage_km), 0)                       AS avg_mileage_km,
        MAX(CAST(f.listing_date AS DATE))                 AS brand_last_seen
    FROM {FACT_LIQUID} f
    JOIN {DIM_CAR} dc ON f.car_sk = dc.car_sk AND dc.__END_AT IS NULL
    WHERE f.listing_date IS NOT NULL
    GROUP BY dc.brand
),
brand_stale AS (
    SELECT
        brand,
        COUNT(*)                    AS stale_combos,
        ROUND(AVG(days_inactive),0) AS avg_days_inactive,
        MAX(days_inactive)          AS max_days_inactive
    FROM {CHURN_TABLE}
    GROUP BY brand
)
SELECT
    t.brand,
    t.total_combos,
    COALESCE(s.stale_combos, 0)                                           AS stale_combos,
    t.distinct_car_sks,
    t.distinct_locations,
    t.total_listings,
    t.avg_price_rub,
    t.avg_mileage_km,
    t.brand_last_seen,
    COALESCE(s.avg_days_inactive, 0)                                      AS avg_days_inactive,
    COALESCE(s.max_days_inactive, 0)                                      AS max_days_inactive,
    ROUND(COALESCE(s.stale_combos,0) / t.total_combos * 100.0, 1)        AS churn_score_pct,
    CASE
        WHEN ROUND(COALESCE(s.stale_combos,0)/t.total_combos*100.0,1) > 80 THEN 'CRITICAL'
        WHEN ROUND(COALESCE(s.stale_combos,0)/t.total_combos*100.0,1) > 50 THEN 'HIGH'
        WHEN ROUND(COALESCE(s.stale_combos,0)/t.total_combos*100.0,1) > 20 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS churn_risk,
    current_timestamp() AS metric_computed_at
FROM brand_totals t
LEFT JOIN brand_stale s ON t.brand = s.brand
ORDER BY churn_score_pct DESC
""")

spark.sql(f'COMMENT ON TABLE {BRAND_CHURN} IS "Brand churn score: pct of car+location combos inactive >180 days"')
brand_cnt = spark.table(BRAND_CHURN).count()
print(f"  Brand churn metrics: {brand_cnt:,} brands scored")
print("\n  Brand churn risk distribution:")
spark.table(BRAND_CHURN).groupBy("churn_risk").count().orderBy(F.desc("count")).show(truncate=False)

## Section 6 — Delta Maintenance

- **VACUUM** — removes unreferenced old files. 168-hour retention = 7-day time travel window.
- **OPTIMIZE** — compacts small files. On Liquid tables, also re-applies clustering.

In [0]:
print("  VACUUM (168-hour retention)...")
for tbl in [FACT_LIQUID, FACT_PART, CHURN_TABLE, BRAND_CHURN]:
    spark.sql(f"VACUUM {tbl} RETAIN 168 HOURS")
    print(f"  VACUUM done: {tbl.split('.')[-1]}")

print("\n  OPTIMIZE...")
spark.sql(f"OPTIMIZE {FACT_LIQUID}")                              # re-clusters by car_sk, listing_year, location_sk
spark.sql(f"OPTIMIZE {FACT_PART} ZORDER BY (car_sk, location_sk)") # re-z-orders
print("  OPTIMIZE complete")

print("\n  Delta History (fact_listings_liquid, last 8 ops):")
display(
    spark.sql(f"DESCRIBE HISTORY {FACT_LIQUID}")
    .select("version","timestamp","operation","operationMetrics")
    .orderBy("version", ascending=False)
    .limit(8)
)

## Section 7 — Final Summary & Strategy Comparison

In [0]:
print("=" * 70)
print("  FINAL SUMMARY")
print("=" * 70)

tables_info = [
    (FACT,        "fact_listings (source)"),
    (FACT_LIQUID, "fact_listings_liquid (Liquid Clustering)"),
    (FACT_PART,   "fact_listings_partitioned (Partition+Z-Order)"),
    (CHURN_TABLE, "agg_stale_inventory (Churn Metric 1)"),
    (BRAND_CHURN, "agg_brand_churn_metrics (Churn Metric 2)"),
    (BENCH_TABLE, "benchmark_results"),
]
print()
for tbl, label in tables_info:
    cnt = spark.table(tbl).count()
    print(f"  {label:<52} {cnt:>10,} rows")

print("\n  Benchmark Results:")
display(spark.table(BENCH_TABLE).orderBy("query"))

print()
total_stale  = spark.table(CHURN_TABLE).count()
critical_cnt = spark.table(CHURN_TABLE).filter("churn_status = 'CRITICAL'").count()
print(f"  Stale inventory combos  : {total_stale:,}")
print(f"  CRITICAL (>365 days)    : {critical_cnt:,}")

print()
print("  Brand churn risk distribution:")
spark.table(BRAND_CHURN).groupBy("churn_risk").count().orderBy(F.desc("count")).show(truncate=False)

print("=" * 70)
print("  STRATEGY RECOMMENDATION")
print("=" * 70)
print()
print("  USE Liquid Clustering when:")
print("    Filters span multiple integer keys (car_sk + location_sk + listing_year)")
print("    Filter combinations are unpredictable (BI / ad-hoc queries)")
print("    You want zero partition management overhead")
print()
print("  USE Partitioned + Z-Order when:")
print("    One column is ALWAYS in the WHERE clause (e.g. listing_year)")
print("    Very high data volume per partition")
print()
print("  NOTE: Queries on brand/model/fuel_type require a JOIN to dim_car")
print("  on car_sk. This is the correct Kimball pattern -- integer FK joins")
print("  are significantly faster than string column scans at scale.")